In [1]:
!pip  install torch transformers pytorch_lightning fugashi ipadic unidic_lite pytorch_lightning  torchmetrics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 66.0 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 51.6 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 695.3/695.3 kB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 71.9 MB/s eta 0:00:00
  Created wheel for ipadic: filename=ipadic-1.0.0-py3-none-any.whl size=13556705 sha256=3f51a57be1c7ca9f2e07cb60ab2a70e68e84243d0e175cca5233ebdf34df8e88
  Stored in directory: /root/.cache/pip/wheels/44/56/37/f543963822b85260c9f948df8fac8c20169c80dc71b24dc407
  Created wheel for unidic_lite: filename=unidic_lite-1.0.8-py3-none-any.whl size=47658818 sha256=01688447ef4ab621e58168a43245dfdc7052345b335aa4c86db23e191a5f82c4
  Stored in directory: /root/.cache/pip/wheels/b7/fd/e9/ea4459b868e6d2

In [2]:
import torch, transformers
print(torch.__version__)
print(transformers.__version__)

2.1.1+cu121
4.35.2


In [9]:
from transformers import AutoTokenizer, AutoModel
import torch
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
import unicodedata
import torch.nn as nn
import numpy as np
import random
import torchmetrics.functional as FM
from torchmetrics.functional import precision, recall, f1_score
from torchmetrics.classification import BinaryF1Score
import unicodedata
import re
import json
from sklearn.model_selection import train_test_split
import random
from collections import Counter


MODEL_PATH = "line-corporation/line-distilbert-base-japanese"
FILE_NAME = "sentences_all.jsonl"

# 再現性のための乱数固定
def set_seed(seed=42):
    random.seed(seed)                          
    np.random.seed(seed)                     
    torch.manual_seed(seed)                             
    torch.cuda.manual_seed_all(seed)           
    torch.backends.cudnn.deterministic = True  
    torch.backends.cudnn.benchmark = False     
SEED=42
set_seed(42)

In [10]:
class GapProjectorLearnable(nn.Module):
    """
    トークン間のギャップを生成
    ギャップの左右 L個・R個のトークンを学習可能な重みで加重平均する。
    """
    def __init__(self, L=2, R=2, mode="concat"):
        super().__init__()
        
        self.L = L
        self.R = R
        
        self.weight_L = nn.Parameter(torch.zeros(self.L))#左側の重み
        self.weight_R = nn.Parameter(torch.zeros(self.R))#右側の重み
        
        self.mode = mode
    def forward(self, x): # x: (B, T, H)bertの出力結果
        B, T, H = x.shape
        device, dtype = x.device, x.dtype
        
        # 左右にパディングしてギャップ位置のスライスを可能にする
        padL = torch.zeros((B, self.L, H), device=device, dtype=dtype)
        padR = torch.zeros((B, self.R, H), device=device, dtype=dtype)
        
        padx = torch.cat([padL, x, padR], dim= 1) #(B, (L+T+R), H)
        
        nor_weight_L = torch.softmax(self.weight_L, dim=0) if self.L > 0 else None
        nor_weight_R = torch.softmax(self.weight_R, dim=0) if self.R > 0 else None
        
        # 左文脈の加重平均
        left = torch.zeros((B, T+1, H), device=device, dtype=dtype)
        for k in range(1, self.L+1):
            left = left + nor_weight_L[k-1] * padx[:, self.L-k : self.L-k+T+1, :]
            
         # 右文脈の加重平均
        right = torch.zeros((B, T+1, H), device=device, dtype=dtype)
        for k in range(self.R):
            right = right + nor_weight_R[k] * padx[:, self.L+k : self.L+k+T+1, :]
        
        if self.mode == "concat":
            y = torch.cat([left, right], dim=-1) #(B, T+1, 2*H)
        else:
            y = (left + right) / 2 #(B, T+1, H)
            
        return y
            
        

In [11]:

class Tokenizer(nn.Module):
    """
    括弧挿入タスク専用のトークナイザ。
    丸括弧を除去した入力と、ギャップ位置のラベルを生成する。
    """
    def __init__(self, model_path):
        super().__init__()
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_path,trust_remote_code=True)
        # self.special_tokens_ids = self.tokenizer.all_special_ids
        
        
        SYMBOL = r"[()（）]"
        self.pat = re.compile(SYMBOL)# 丸括弧（全角・半角）にマッチするパターン
        
        self.symbol_dict = {"(":1, ")":2}
        
        
    def make_label_and_encode(self, text, max_length=128):

        text = self.normalize_text(text)

        # 括弧ありで tokenize（ラベル生成用）
        encode_A = self.tokenizer(
            text,
            max_length=max_length,
            add_special_tokens=True, # [CLS]や[SEP]をつける
            return_tensors="pt"
        )

        input_ids_A = encode_A["input_ids"][0]
        tokens_A = self.tokenizer.convert_ids_to_tokens(input_ids_A)
        

        # トークン列から括弧を除去してモデルへの入力を作成
        tokens_B = []
        for tok in tokens_A:
            if any(p in tok for p in ["(", ")", "（", "）"]):
                continue
            tokens_B.append(tok)

        T = len(tokens_B)
        gap_labels = [[0.0, 0.0] for _ in range(T + 1)]#(T+1)個のギャップ

        #括弧トークンの位置からギャップラベルを決定
        b_index = 0
        for tok in tokens_A:
            if "(" in tok or "（" in tok:
                gap_labels[b_index][0] = 1.0
            elif ")" in tok or "）" in tok:
                gap_labels[b_index][1] = 1.0
            else:
                b_index += 1

        # トークンidに変換してパディング
        input_ids_B = self.tokenizer.convert_tokens_to_ids(tokens_B)
        pad_id = self.tokenizer.pad_token_id
        if len(input_ids_B) < max_length:
            input_ids_B += [pad_id] * (max_length - len(input_ids_B))
        else:
            input_ids_B = input_ids_B[:max_length]

        # ギャップラベルもmaxLength + 1にパディング
        target_len = max_length + 1
        if len(gap_labels) < target_len:
            gap_labels += [[0.0,0.0]] * (target_len - len(gap_labels))
        else:
            gap_labels = gap_labels[:target_len]

        return {
            "input_ids": torch.tensor(input_ids_B),
            "attention_mask": torch.tensor(
                [1 if i != pad_id else 0 for i in input_ids_B]
            ),
            "labels": torch.tensor(gap_labels, dtype=torch.float)
        }

    def make_encode(self, text, max_length = None, return_tensors=None):
        """推論時用のエンコード（ラベルなし）"""
        encode = self.tokenizer(text,
                                max_length=max_length,
                                padding="max_length" if max_length else "do_not_pad",
                                truncation=True if max_length else None)
        
        if return_tensors == "pt":
            encode = {k:torch.tensor([v]) for k, v in encode.items()}
            
        return encode
        
    def normalize_text(self, text):
        """文章を正規化"""
        return unicodedata.normalize("NFKC", text)
        

In [12]:
tok = Tokenizer(MODEL_PATH)

In [13]:
class NewDataset(Dataset):
    def __init__(self, data, tokenizer):
        """
        括弧あり・なしを交互にサンプリングするデータセット。

        """
        super().__init__()
        self.data = data
        self.no_paren_data = [d for d in data if d["label"] == 0]#括弧なし
        self.paren_data = [d for d in data if d["label"] == 1]#括弧あり
        self.tokenizer = tokenizer
    
    def __getitem__(self,idx):
        # 偶数indexは括弧なし、奇数indexは括弧ありを返す
        if idx % 2 == 0:
            d = self.no_paren_data[idx % len(self.no_paren_data)]
        else:
            d = self.paren_data[idx % len(self.paren_data)]
        
        encode = self.tokenizer.make_label_and_encode(d["text"])
        
        return encode
    
    def __len__(self):
         # 少ない方に合わせる
        return min(len(self.no_paren_data), len(self.paren_data)) * 2

In [14]:

USE_SENTENCE_NUM = 100000

# データ読み込み
data = []
with open(FILE_NAME, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

# 括弧なし・括弧あり文を同数にする
data_no = [d for d in data if d["label"] == 0][:USE_SENTENCE_NUM]
data_pa = [d for d in data if d["label"] == 1][:USE_SENTENCE_NUM]

data = data_no + data_pa

#データをシャッフルする
shuffled_data = random.sample(data, k=len(data))
labels_ratio = [d["label"] for d in shuffled_data]

#学習データ、検証データ、テストデータに分ける(6:2:2)
train_val_data, test_data = train_test_split(
            shuffled_data,
            test_size = 0.2,
            random_state = SEED,
            stratify = labels_ratio)

train_val_labels_ratio = [d["label"] for d in train_val_data]

train_data, val_data = train_test_split(
            train_val_data,
            test_size = 0.25,
            random_state = SEED,
            stratify=train_val_labels_ratio)

#データサイズとクラスでの比率確認
train_labels = [d["label"] for d in train_data] 
val_labels = [d["label"] for d in val_data] 
test_labels = [d["label"] for d in test_data] 

print(f"trainデータサイズ {len(train_labels)}")
print(f"validデータサイズ{len(val_labels)}")
print(f"testデータサイズ{len(test_labels)}")
train = Counter(train_labels)
val = Counter(val_labels)
test = Counter(test_labels)

print(f"学習データの比率 | 括弧なし {train[0] / (train[0] + train[1]):.3f} | 括弧あり {train[1] / (train[0] + train[1]):.3f}")
print(f"検証データの比率 | 括弧なし {val[0] / (val[0] + val[1]):.3f} | 括弧あり {val[1] / (val[0] + val[1]):.3f}")
print(f"テストデータの比率 | 括弧なし {test[0] / (test[0] + test[1]):.3f} | 括弧あり {test[1] / (test[0] + test[1]):.3f}")

trainデータサイズ 120000
validデータサイズ40000
testデータサイズ40000
学習データの比率 | 括弧なし 0.500 | 括弧あり 0.500
検証データの比率 | 括弧なし 0.500 | 括弧あり 0.500
テストデータの比率 | 括弧なし 0.500 | 括弧あり 0.500


In [15]:
#データセット作成
train_dataset = NewDataset(train_data, tok)
val_dataset = NewDataset(val_data, tok)
test_dataset = NewDataset(test_data, tok)

In [16]:
#DataLoader作成
train_dataloader = DataLoader(dataset=train_dataset, batch_size=128, shuffle=True)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=256)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=256)

In [17]:
class GapClassification(pl.LightningModule):
    """
    BERTを用いて丸括弧の挿入位置を予測するモデル。
    トークン間のギャップごとに、開き括弧と閉じ括弧の挿入をバイナリ分類で予測する。
    """
    def __init__(self, model_path, tok,L=5, R=5, lr=1e-5, mode="concat"):
        super().__init__()
        self.save_hyperparameters()
        
        self.model = AutoModel.from_pretrained(model_path)
            
        self.gap_pro = GapProjectorLearnable(L=L, R=R, mode=mode)
        
        self.val_open_f1 = BinaryF1Score()
        self.val_close_f1 = BinaryF1Score()
        
        #modeに応じて分類ヘッドの入力サイズを決定
        if mode == "concat":
            input_size = self.model.config.hidden_size * 2
        else:
            input_size = self.model.config.hidden_size
            
        #開き括弧の予測ヘッド
        self.head_open = nn.Sequential(
                nn.Dropout(0.1),
                nn.Linear(input_size, 1)
        )
        #閉じ括弧の予測ヘッド(開き括弧の確率を特徴量として追加)
        self.head_close = nn.Sequential(
                nn.Dropout(0.1),
                nn.Linear(input_size+1, 1)
        )
        
        self.loss_fct = nn.BCEWithLogitsLoss(reduction="none")
        self.tokenizer = tok.tokenizer
        
    def forward(self, batch):
        
        output = self.model(**batch)
        scores = output.last_hidden_state # (B, T, H)
        
        #ギャップ特徴量を生成
        gap_feats = self.gap_pro(scores)
        
        #開き括弧の予測
        open_logits = self.head_open(gap_feats)
        open_probs = torch.sigmoid(open_logits)
        
        # 閉じ括弧の予測（開き括弧の確率を入力に追加して対応関係を考慮）
        close_input = torch.cat([gap_feats, open_probs], dim=-1)
        close_logits = self.head_close(close_input)
        
        
        return torch.cat([open_logits, close_logits], dim=-1)
        
        
    def _make_gap_mask(self, attention_mask):
        """CLSトークンを除外したgap用マスクを作成"""
        gap_mask = attention_mask.clone()
        gap_mask[:, 0] = 0  # CLS除外
        gap_mask = torch.cat(
            [gap_mask, torch.zeros_like(gap_mask[:, :1])],
            dim=1
        )
        return gap_mask.unsqueeze(-1)  # (B, T+1, 1)
    
    def training_step(self, batch , batch_idx):
        
        labels = batch.pop("labels")  # (B, T+1, 2)
        attention_mask = batch["attention_mask"]  # (B, T)
        
        logits = self(batch)
        gap_mask = self._make_gap_mask(attention_mask)
        
        loss_raw = self.loss_fct(logits, labels.float())
        loss = (loss_raw * gap_mask).sum() / (gap_mask.sum()*2)

        self.log("train_loss", loss)
        return loss
        

    def validation_step(self, batch, batch_idx):

        labels = batch.pop("labels")
        attention_mask = batch["attention_mask"]

        logits = self(batch)
        probs = torch.sigmoid(logits)
        gap_mask = self._make_gap_mask(attention_mask)
        
        loss_raw = self.loss_fct(logits, labels.float())
        loss = (loss_raw * gap_mask).sum() / (gap_mask.sum()*2)
        self.log("val_loss", loss)
        
        # 閾値0.5で予測
        mask = gap_mask.squeeze(-1) == 1
        open_pred = (probs[...,0] > 0.5)
        close_pred = (probs[...,1] > 0.5)
        

        # 全体で開き・閉じ両方が正解かを評価
        pair_correct = (open_pred == labels[..., 0].bool()) & (close_pred == labels[..., 1].bool())
        is_correct_or_ignored = pair_correct | (~mask)
        sequence_perfect = is_correct_or_ignored.all(dim=1).float()
        
        self.val_open_f1.update(probs[...,0][mask], labels[...,0][mask].long())
        self.val_close_f1.update(probs[...,1][mask], labels[...,1][mask].long())

        self.log("val_open_f1", self.val_open_f1, on_step=False, on_epoch=True)
        self.log("val_close_f1", self.val_close_f1, on_step=False, on_epoch=True)
        self.log("val_sequence_em", sequence_perfect.mean(), on_step=False, on_epoch=True)
        return loss
    
    def test_step(self, batch, batch_idx):
        
        labels = batch.pop("labels")          # (B, T+1, 2)
        attention_mask = batch["attention_mask"] # (B, T)

        logits = self(batch)
        probs = torch.sigmoid(logits)
        gap_mask = self._make_gap_mask(attention_mask)

        loss_raw = self.loss_fct(logits, labels)
        loss = (loss_raw * gap_mask).sum() / (gap_mask.sum() + 1e-8)
        self.log("test_loss", loss)

        preds = (probs > 0.5).float()
        mask = gap_mask.squeeze(-1) == 1

        #開き括弧の評価
        open_pred = preds[..., 0][mask]
        open_label = labels[..., 0][mask]
        
        open_f1 = f1_score(open_pred, open_label, task="binary")
        open_precision = precision(open_pred, open_label, task="binary")
        open_recall = recall(open_pred, open_label, task="binary")

        self.log("test_open_f1", open_f1)
        self.log("test_open_precision", open_precision)
        self.log("test_open_recall", open_recall)

        #閉じ括弧の評価
        close_pred = preds[..., 1][mask]
        close_label = labels[..., 1][mask]

        close_f1 = f1_score(close_pred, close_label, task="binary")
        close_precision = precision(close_pred, close_label, task="binary")
        close_recall = recall(close_pred, close_label, task="binary")

        self.log("test_close_f1", close_f1)
        self.log("test_close_precision", close_precision)
        self.log("test_close_recall", close_recall)

        # 偽陽性率
        fp_open = open_pred[(open_label == 0)].float().mean() if (open_label == 0).sum() > 0 else torch.tensor(0.0)
        fp_close = close_pred[(close_label == 0)].float().mean() if (close_label == 0).sum() > 0 else torch.tensor(0.0)
        self.log("test_fp_open", fp_open)
        self.log("test_fp_close", fp_close)

        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)


        


In [ ]:
#記録の設定
logger_csv = pl.loggers.CSVLogger("outputs", name="Lightning_logs_csv2")
logger_tb = pl.loggers.TensorBoardLogger("outputs", name="Lightning_logs_tb2")

#ベストモデルの保存設定
check_point = pl.callbacks.ModelCheckpoint(
                monitor = "val_loss",
                mode = "min",
                save_top_k = 1,
                save_weights_only = True,
                dirpath = "model/"
                )

#トレーナーの設定
trainer = pl.Trainer(
        max_epochs = 5,
        callbacks = [check_point],
        accelerator = "gpu",
        devices = 1,
        logger = [logger_csv, logger_tb])

# モデルの初期化
model = GapClassification(model_path = MODEL_PATH, tok=tok)

#学習・評価
trainer.fit(model, train_dataloader, val_dataloader)
result = trainer.test(dataloaders = test_dataloader)
print(result)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.11/dist-packages/pytorch_lightning/utilities/parsing.py:213: Attribute 'tok' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['tok'])`.


config.json:   0%|          | 0.00/509 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/275M [00:00<?, ?B/s]

2026-04-28 07:37:10.983881: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-28 07:37:10.984030: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-28 07:37:11.090341: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 07:37:11.309796: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-28 07:37:13.481109: W tensorflow/compiler/tf2

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=5` reached.
/usr/local/lib/python3.11/dist-packages/pytorch_lightning/trainer/connectors/checkpoint_connector.py:149: `.test(ckpt_path=None)` was called without a model. The best model of the previous `fit` call will be used. You can pass `.test(ckpt_path='best')` to use the best model or `.test(ckpt_path='last')` to use the last model. If you pass a value, this warning will be silenced.
Restoring states from the checkpoint path at /notebooks/model/epoch=2-step=2814.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /notebooks/model/epoch=2-step=2814.ckpt
/usr/local/lib/python3.11/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]